### Install the DQX dependency

**Purpose:** Make the DQX library available to this notebook runtime.

**Inputs:** The project dependency specification.

**Outputs:** The DQX package installed for the current notebook session.

**Why it matters:** Quality monitoring depends on the same rule engine used by the production-style Job task.

In [0]:
%pip install databricks-labs-dqx

### Load DQX for quality monitoring

**Purpose:** Install the DQX dependency used by the quality-monitoring branch.

**Inputs:** Project DQX dependency configuration.

**Outputs:** A runtime capable of applying the configured quality checks.

**Why it matters:** Quality monitoring is a required terminal branch before final validation.

In [0]:
%restart_python

### Restart after dependency installation

**Purpose:** Restart the notebook Python process so the installed DQX package is importable.

**Inputs:** The package installed in the previous cell.

**Outputs:** A clean interpreter session.

**Why it matters:** DQX initialization must use the same dependency state as the monitoring task.

In [0]:
%run ./00_setup

### Load the DQX monitoring context

**Purpose:** Apply the shared setup and prepare the target namespace for DQX.

**Inputs:** Demo2 Olist catalog, schema, and runtime configuration.

**Outputs:** The Spark and Unity Catalog context used for quality checks.

**Why it matters:** Quality results must be written beside the Gold data they describe.

In [0]:
# Purpose: verify the installed DQX version and initialize
# the DQX engine using Databricks notebook authentication.

from importlib.metadata import version

from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

dqx_version = version(
    "databricks-labs-dqx"
)

workspace_client = WorkspaceClient()
dq_engine = DQEngine(workspace_client)

print("DQX version:", dqx_version)
print("DQX engine initialized successfully")
print("Catalog:", catalog)
print("Schema:", schema)

### Initialize and verify DQX

**Purpose:** Confirm the installed DQX version and create the authenticated quality engine.

**Inputs:** Databricks notebook authentication, Spark session, catalog, and schema.

**Outputs:** A configured `DQEngine` and visible runtime version.

**Why it matters:** Explicit initialization makes quality execution auditable and repeatable.

In [0]:
# Purpose: load the Gold order-item fact table that DQX will monitor.

dqx_input_table = (
    f"{catalog}.{schema}.gold_fact_order_items"
)

if not spark.catalog.tableExists(dqx_input_table):
    raise RuntimeError(
        f"DQX input table does not exist: {dqx_input_table}"
    )

dqx_input_df = spark.table(
    dqx_input_table
)

dqx_input_rows = dqx_input_df.count()

print("DQX input table:", dqx_input_table)
print("DQX input rows:", dqx_input_rows)

display(
    dqx_input_df.limit(10)
)

### Load the Gold fact for quality checks

**Purpose:** Read and verify the Gold order-item fact that DQX will inspect.

**Inputs:** `gold_fact_order_items` in the configured catalog and schema.

**Outputs:** The DQX input DataFrame and its row count.

**Why it matters:** A missing or empty input must stop quality monitoring before audit tables are written.

### Preserve the notebook cell boundary

**Purpose:** Keep the intentional empty separator cell in the notebook layout.

**Inputs:** No executable inputs.

**Outputs:** No data side effect.

**Why it matters:** The cell remains a presentation boundary and is not counted as executable logic.

In [0]:
# Purpose: define explicit DQX rules for required fields,
# monetary ranges, and surrogate-key uniqueness.

from databricks.labs.dqx.engine import DQEngine

required_string_columns = [
    "order_item_sk",
    "order_id",
    "customer_sk",
    "product_sk",
    "seller_sk",
    "customer_id",
    "product_id",
    "seller_id",
    "order_status",
]

# Create one explicit rule for every required string column.
dqx_checks = [
    {
        "name": f"{column}_required",
        "criticality": "error",
        "check": {
            "function": "is_not_null_and_not_empty",
            "arguments": {
                "column": column,
                "trim_strings": True,
            },
        },
    }
    for column in required_string_columns
]

# Add rules that are not string-based.
dqx_checks.extend(
    [
        {
            "name": "date_sk_required",
            "criticality": "error",
            "check": {
                "function": "is_not_null",
                "arguments": {
                    "column": "date_sk",
                },
            },
        },
        {
            "name": "valid_price_range",
            "criticality": "error",
            "check": {
                "function": "is_in_range",
                "arguments": {
                    "column": "price",
                    "min_limit": 0,
                    "max_limit": 100000,
                },
            },
        },
        {
            "name": "valid_freight_range",
            "criticality": "error",
            "check": {
                "function": "is_in_range",
                "arguments": {
                    "column": "freight_value",
                    "min_limit": 0,
                    "max_limit": 100000,
                },
            },
        },
        {
            "name": "valid_item_total_range",
            "criticality": "error",
            "check": {
                "function": "is_in_range",
                "arguments": {
                    "column": "item_total_value",
                    "min_limit": 0,
                    "max_limit": 200000,
                },
            },
        },
        {
            "name": "unique_order_item_surrogate_key",
            "criticality": "error",
            "check": {
                "function": "is_unique",
                "arguments": {
                    "columns": [
                        "order_item_sk",
                    ],
                },
            },
        },
    ]
)

dqx_rule_validation = DQEngine.validate_checks(
    dqx_checks
)

print("Defined DQX rules:", len(dqx_checks))
print("DQX rule validation:", dqx_rule_validation)

if dqx_rule_validation.has_errors:
    raise RuntimeError(
        "Invalid DQX rule configuration: "
        + str(dqx_rule_validation.errors)
    )

print("PASS: all DQX rules are valid")

### Define the DQX rule contract

**Purpose:** Build explicit required-field, range, and uniqueness checks for Gold order items.

**Inputs:** Gold fact column names and DQX rule definitions.

**Outputs:** A validated list of 14 critical quality rules.

**Why it matters:** The rule contract protects keys, measures, and analytical grain before records are classified.

In [0]:
# Purpose: apply the validated DQX rules and split the Gold fact
# into valid records and records requiring quarantine.

dqx_valid_df, dqx_quarantine_df = (
    dq_engine.apply_checks_by_metadata_and_split(
        dqx_input_df,
        dqx_checks,
    )
)

dqx_valid_rows = dqx_valid_df.count()
dqx_quarantine_rows = dqx_quarantine_df.count()

dqx_reconciled_rows = (
    dqx_valid_rows
    + dqx_quarantine_rows
)

dqx_reconciliation_pass = (
    dqx_reconciled_rows
    == dqx_input_rows
)

print("DQX input rows:", dqx_input_rows)
print("DQX valid rows:", dqx_valid_rows)
print("DQX quarantine rows:", dqx_quarantine_rows)
print("DQX reconciled rows:", dqx_reconciled_rows)

print(
    "DQX reconciliation:",
    "PASS" if dqx_reconciliation_pass else "FAIL",
)

print("Quarantined records and DQX error details:")

display(
    dqx_quarantine_df.limit(20)
)

### Split valid and quarantined records

**Purpose:** Apply the validated DQX rules and separate passing records from violations.

**Inputs:** Gold fact DataFrame and the 14 configured checks.

**Outputs:** Valid and quarantine DataFrames plus reconciliation counts.

**Why it matters:** The input must reconcile to valid plus quarantined rows before audit outputs are trusted.

In [0]:
# Purpose: save the current DQX quarantine result.
# This table is overwritten because it represents the latest snapshot.

from pyspark.sql import functions as F

dqx_quarantine_table = (
    f"{catalog}.{schema}."
    "dqx_gold_fact_order_items_quarantine"
)

dqx_quarantine_output_df = (
    dqx_quarantine_df
    .withColumn(
        "_dqx_checked_by",
        F.expr("session_user()"),
    )
    .withColumn(
        "_dqx_checked_at",
        F.current_timestamp(),
    )
)

(
    dqx_quarantine_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dqx_quarantine_table)
)

print(
    "DQX quarantine table:",
    dqx_quarantine_table,
)

print(
    "Saved quarantine rows:",
    spark.table(dqx_quarantine_table).count(),
)

### Write the DQX quarantine snapshot

**Purpose:** Persist the current invalid-record set with audit timestamps.

**Inputs:** The quarantine DataFrame and current user/session metadata.

**Outputs:** `dqx_gold_fact_order_items_quarantine`, overwritten as the latest snapshot.

**Why it matters:** Operators need the failing rows and check context for remediation.

In [0]:
# Purpose: save a historical DQX monitoring summary for every run.

dqx_quality_pass = (
    dqx_reconciliation_pass
    and dqx_quarantine_rows == 0
)

dqx_quality_status = (
    "PASS"
    if dqx_quality_pass
    else "FAIL"
)

dqx_audit_df = spark.createDataFrame(
    [
        (
            dqx_input_table,
            dqx_input_rows,
            dqx_valid_rows,
            dqx_quarantine_rows,
            dqx_reconciled_rows,
            len(dqx_checks),
            dqx_quality_status,
        )
    ],
    """
        input_table STRING,
        input_rows LONG,
        valid_rows LONG,
        quarantine_rows LONG,
        reconciled_rows LONG,
        configured_rules INT,
        status STRING
    """,
)

dqx_audit_df = (
    dqx_audit_df
    .withColumn(
        "checked_by",
        F.expr("session_user()"),
    )
    .withColumn(
        "_checked_at",
        F.current_timestamp(),
    )
)

dqx_audit_table = (
    f"{catalog}.{schema}."
    "dqx_gold_fact_order_items_audit"
)

(
    dqx_audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(dqx_audit_table)
)

display(
    spark.table(dqx_audit_table)
    .orderBy(F.col("_checked_at").desc())
    .limit(10)
)

if not dqx_quality_pass:
    raise RuntimeError(
        f"DQX validation failed with "
        f"{dqx_quarantine_rows} quarantined rows."
    )

print("SUCCESS: DQX quality monitoring passed")

### Record DQX audit status

**Purpose:** Append the run-level DQX summary and fail the task if quality reconciliation or quarantine policy fails.

**Inputs:** Input, valid, quarantine, reconciled row counts, and configured rule count.

**Outputs:** `dqx_gold_fact_order_items_audit` plus a strict PASS/FAIL task result.

**Why it matters:** This terminal quality result gates final validation and downstream publication.